In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 1    
MON_GAP_1 = 0
MON_GAP_2 = 0
HAS_BOOSTER = 1  
HORIZON_NUIT = 5

💻 Environnement Local (VS Code) détecté.


In [2]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE ET RECOMMANDATION
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID}...")

reco, wr, ev, q_table_jour = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques synchronisées sur le Drive pour la version Mobile.")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION OFFICIELLE : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 1...
✅ Terminé ! 5 abaques synchronisées sur le Drive pour la version Mobile.

🎯 RECOMMANDATION OFFICIELLE : 1 (Dom) (Safe)
   Espérance de Victoire (WR) : 33.56%


In [ ]:
# ==========================================
# 1bis. DÉTAIL COMPLET DU MATCH DU JOUR
# ==========================================
# La Q-table stocke les 9 win rates (3 outcomes × 3 modes booster).
# q_table_jour[g1, g2, outcome, mode] avec mode : 0=sans booster, 1=keep (parier+garder x2), 2=use (parier+x2)
# On affiche tout pour permettre l'arbitrage manuel via le potentiel de score exact
# (non modélisé) quand deux win rates sont très proches.

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]

g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(MON_GAP_1)))) + GAP_OFFSET
g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(MON_GAP_2)))) + GAP_OFFSET

detail = q_table_jour[g1_idx, g2_idx, :, :]   # shape (3, 3) : [outcome, mode]

# Colonne "Safe" = keep si on a le booster, sinon mode sans booster
col_safe = 1 if HAS_BOOSTER else 0
best_safe_idx = int(np.argmax(detail[:, col_safe]))

print("\n" + "=" * 66)
print("📊 DÉTAIL COMPLET DU MATCH DU JOUR (tous les outcomes)")
print("=" * 66)
print(f"{'Outcome':<10} | {'EV (pts)':>9} | {'WR Safe':>9} | {'WR x2':>9} | {'Δ x2':>8}")
print("-" * 66)
for a in range(3):
    wr_safe = detail[a, col_safe]
    wr_x2   = detail[a, 2]
    delta   = wr_x2 - wr_safe
    star    = "  ⭐" if a == best_safe_idx else ""
    print(f"{noms_choix[a]:<10} | {ev[a]:>9.1f} | {wr_safe*100:>8.2f}% | {wr_x2*100:>8.2f}% | {delta*100:>+7.2f}%{star}")
print("-" * 66)
print("💡 Si deux WR sont quasi identiques, privilégie l'outcome au meilleur")
print("   potentiel de score exact (bonus non pris en compte par le modèle).")

In [3]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 1 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.56% | WR(x2): 29.26%
  ⚪ Maintien (Inchangé)       | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.56% | WR(x2): 29.26%
  🟢 Avance (+60 pts/match)    | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.56% | WR(x2): 29.26%
------------------------------------------------------------------------------------------
▶️ MATCH 2 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :  -60,  -60 | Reco : 1 (Dom) (Safe) | WR(Safe): 31.26% | WR(x2): 27.39%
  ⚪ Maintien (Inchangé)       | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 35.32% | WR(x2): 31.56%
  🟢 Avance (+60 pts/match)    | Gaps proj. :   60,   60 | Reco : 1 (Dom) (Safe) | WR(Safe): 39.96% | WR(x2): 36.28%
-------------------------------